# scPRINT2 - Gene Regulatory Network Inference on T Cell Data

This notebook runs GRN inference using scPRINT2 on the T cell dataset.
It follows the same data preparation pipeline used for embedding generation,
then uses `GNInfer` to extract attention-based gene networks per cell type.

**Prerequisites:**
- LaminDB instance already initialized and populated (from embedding notebook)
- `small-v2.ckpt` checkpoint already downloaded
- Raw T cell h5ad already downloaded to `./my_data_folder/mydata.h5ad`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!pip install scprint2 lamindb bionty scdataloader scanpy anndata scipy numpy torch huggingface_hub grnndata bengrn

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import torch
import numpy as np
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import bionty as bt
import lamindb as ln

from huggingface_hub import hf_hub_download
from anndata.utils import make_index_unique
from scdataloader.utils import load_genes
from grnndata import utils as grnutils

torch.set_float32_matmul_precision('medium')

print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

torch version: 2.8.0+cu128
CUDA available: True


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
CKPT_PATH   = './small-v2.ckpt'
H5AD_PATH   = './my_data_folder/mydata.h5ad'
GRN_OUT_DIR = './grn_outputs'

os.makedirs(GRN_OUT_DIR, exist_ok=True)

# The column in adata.obs that defines cell groups for GRN inference.
# Change this to whatever grouping variable you want GRNs for
# (e.g. 'stimulation', 'cd_status', or scPRINT's predicted 'conv_pred_cell_type_ontology_term_id')
CELL_TYPE_COL = 'stimulation'

# How many genes to use per GRN (more = richer network, slower)
NUM_GENES = 2000

# Max cells per cell type to use (keep manageable on a single GPU)
MAX_CELLS = 300

## Step 1: Connect to LaminDB

We reconnect to the same instance used during embedding generation.
This is already populated with ontologies and all 16 organism gene tables.

In [ ]:
import lamindb as ln
import shutil

shutil.rmtree('/content/lamindb-storage', ignore_errors=True)

ln.setup.init(
    storage='/content/lamindb-storage',
    name='scprintGRN',
    modules='bionty'
)

! using anonymous user (to identify, call: lamin login)
→ initialized lamindb: anonymous/scprintGRN


/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:218: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  isinstance(v, types.ModuleType)
/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:225: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif hasattr(v, "__module__") and getattr(v, "__module__", None):
/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:226: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  class_module = v.__module__


In [ ]:
from scdataloader.utils import populate_my_ontology, load_genes
populate_my_ontology()
load_genes(organisms='NCBITaxon:9606')

... synchronizing df_all__cl__2025-12-17__CellType.parquet: 100.0%
... synchronizing df_vertebrates__ensembl__release-114__Organism.parquet: 100.0%
! ambiguous validation in Bionty for 1 record: 'sheep'
! ambiguous validation in Bionty for 1 record: 'sheep'
! ontology ID BFO:0000020 not found in DataFrame
... synchronizing df_human__hancestro__2025-10-14__Ethnicity.parquet: 100.0%
→ starting creation of 18326 ExperimentalFactor records in batches of 10000
→ starting creation of 16430 ExperimentalFactor_parents records in batches of 10000
! you are trying to create a record with name='unknown' but a record with similar name exists: 'fever of unknown origin'. Did you mean to load it?
... synchronizing df_all__uberon__2025-12-04__Tissue.parquet: 100.0%
→ starting creation of 15770 Tissue records in batches of 10000
→ starting creation of 42140 Tissue_parents records in batches of 10000
... synchronizing df_human__hsapdv__2025-01-23__DevelopmentalStage.parquet: 100.0%
... synchronizing df_

,uid,symbol,biotype,is_locked,branch_id,organism_id,mt,ribo,hb,organism
ensembl_gene_id,,,,,,,,,,
ENSG00000000003,1uTi9dROoaN5rR,TSPAN6,protein_coding,False,1,1,False,False,False,NCBITaxon:9606
ENSG00000000005,2FhduD7Z97Uvq1,TNMD,protein_coding,False,1,1,False,False,False,NCBITaxon:9606
ENSG00000000419,4eC1wUNJAO2s7T,DPM1,protein_coding,False,1,1,False,False,False,NCBITaxon:9606
ENSG00000000457,5Zug63FETk4pJF,SCYL3,protein_coding,False,1,1,False,False,False,NCBITaxon:9606
ENSG00000000460,3wEqb6eOTZzC42,FIRRM,protein_coding,False,1,1,False,False,False,NCBITaxon:9606
...,...,...,...,...,...,...,...,...,...,...
ENSG00000310564,5yniswY1TXg2WF,None,transcribed_unitary_pseudogene,False,1,1,False,False,False,NCBITaxon:9606
ENSG00000310566,7RDV4hiXMXTL8E,None,lncRNA,False,1,1,False,False,False,NCBITaxon:9606
ENSG00000310567,4rfaQoGumpscO7,None,lncRNA,False,1,1,False,False,False,NCBITaxon:9606


In [ ]:
import lamindb as ln
ln.connect('scprintGRN')

→ doing nothing, already connected lamindb: anonymous/scprintGRN
→ connected lamindb: anonymous/scprintGRN


/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:218: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  isinstance(v, types.ModuleType)
/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:225: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif hasattr(v, "__module__") and getattr(v, "__module__", None):
/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:226: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  class_module = v.__module__


In [ ]:
print(bt.Organism.df().head())

               uid     name      ontology_id  abbr synonyms description  \
id                                                                        
3   5HWRj1ODkb6V1S  unknown          unknown  None     None        None   
2   3RZqbcSLWNGQGZ    mouse  NCBITaxon:10090  None     None        None   
1   1dpCL6TduFJ3AP    human   NCBITaxon:9606  None     None        None   

   scientific_name  is_locked                       created_at  branch_id  \
id                                                                          
3             None      False 2026-04-13 06:24:13.503000+00:00          1   
2     Mus musculus      False 2026-04-13 06:24:13.492000+00:00          1   
1     Homo sapiens      False 2026-04-13 06:24:13.492000+00:00          1   

    space_id  created_by_id run_id  source_id  
id                                             
3          1              3   None        NaN  
2          1              3   None       34.0  
1          1              3   None       34.0 

/tmp/ipykernel_9049/2657995416.py:1: DeprecationWarning: Use to_dataframe instead of df, df will be removed in the future.
  print(bt.Organism.df().head())


## Step 2: Load and Prepare Data

Same pipeline as the embedding notebook:
1. Load raw h5ad
2. Add required metadata columns
3. Compute PCA + neighbors
4. Convert gene symbols to Ensembl IDs
5. Expand to the full 85,720-gene universe expected by the Collator

In [ ]:
#!/usr/bin/env python3
"""
Load T-cell data from Google Drive
"""

import os
from google.colab import drive

print("=" * 70)
print("LOADING T-CELL DATA FROM GOOGLE DRIVE")
print("=" * 70)
print()

# Mount Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✓ Drive already mounted")

# Configuration
H5AD_PATH = "/content/drive/MyDrive/benchmarking_paper/datasets/tcell_with_embeddings_filtered.h5ad"

# Step 1: Verify file exists
print()
print("Step 1: Checking file...")
if not os.path.exists(H5AD_PATH):
    raise FileNotFoundError(
        f"✗ File not found: {H5AD_PATH}\n"
        f"  Check the path or make sure Drive is mounted correctly."
    )

h5ad_size = os.path.getsize(H5AD_PATH) / (1024**3)  # GB
print(f"✓ File found: {H5AD_PATH}")
print(f"✓ Size: {h5ad_size:.2f} GB")

# Step 2: Validate
print()
print("Step 2: Validating file...")
try:
    import scanpy as sc
    adata = sc.read_h5ad(H5AD_PATH, backed='r')
    print(f"✓ Valid! {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    print(f"  obs columns: {adata.obs.columns.tolist()}")
    del adata
except Exception as e:
    print(f"✗ File validation failed: {e}")
    raise

print()
print("=" * 70)
print("✓ READY!")
print("=" * 70)
print(f"\nData path: {H5AD_PATH}")
print("\nLoad it with: adata = sc.read_h5ad(H5AD_PATH)")

LOADING T-CELL DATA FROM GOOGLE DRIVE

✓ Drive already mounted

Step 1: Checking file...
✓ File found: /content/drive/MyDrive/benchmarking_paper/datasets/tcell_with_embeddings_filtered.h5ad
✓ Size: 0.38 GB

Step 2: Validating file...
✓ Valid! 47,726 cells × 2,000 genes
  obs columns: ['CD_Status', 'Stimulation', 'nUMI', 'leiden']

✓ READY!

Data path: /content/drive/MyDrive/benchmarking_paper/datasets/tcell_with_embeddings_filtered.h5ad

Load it with: adata = sc.read_h5ad(H5AD_PATH)


In [ ]:
print('Loading data...')
adata = sc.read_h5ad(H5AD_PATH)
print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print('obs columns:', adata.obs.columns.tolist())

Loading data...
Loaded: 47,726 cells x 2,000 genes
obs columns: ['CD_Status', 'Stimulation', 'nUMI', 'leiden']


In [ ]:
# Add required scPRINT metadata columns if missing
required_cols = {
    'organism_ontology_term_id':                'NCBITaxon:9606',
    'assay_ontology_term_id':                   'EFO:0009922',
    'self_reported_ethnicity_ontology_term_id': 'unknown',
    'disease_ontology_term_id':                 'PATO:0000461',
    'sex_ontology_term_id':                     'unknown',
}
for col, default in required_cols.items():
    if col not in adata.obs.columns:
        adata.obs[col] = default
        print(f'  Added: {col} = {default}')

# Compute PCA + neighbors (needed for Embedder; GNInfer also benefits)
if 'neighbors' not in adata.uns:
    print('Computing PCA + neighbors...')
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
    print('Done.')
else:
    print('Neighbors already present.')

  Added: organism_ontology_term_id = NCBITaxon:9606
  Added: assay_ontology_term_id = EFO:0009922
  Added: self_reported_ethnicity_ontology_term_id = unknown
  Added: disease_ontology_term_id = PATO:0000461
  Added: sex_ontology_term_id = unknown
Neighbors already present.


In [ ]:
# ── Gene symbol -> Ensembl ID conversion ─────────────────────────────────────
# GNInfer requires Ensembl IDs in var_names, same as the Embedder
import pandas as pd

gene_df = bt.Gene.filter(organism__ontology_id='NCBITaxon:9606').to_dataframe(limit=None)
symbol_to_ensembl = (
    gene_df
    .dropna(subset=['symbol', 'ensembl_gene_id'])
    .set_index('symbol')['ensembl_gene_id']
    .to_dict()
)

new_var_names = [symbol_to_ensembl.get(g, None) for g in adata.var_names]
mapped = sum(1 for x in new_var_names if x is not None)
print(f'Mapped {mapped}/{adata.n_vars} genes to Ensembl IDs')

Mapped 1191/2000 genes to Ensembl IDs


In [ ]:
CKPT_PATH   = './small-v2.ckpt'
H5AD_PATH   = './my_data_folder/mydata.h5ad'
GRN_OUT_DIR = './grn_outputs'
print('Loading model to get gene universe...')
from scprint2 import scPRINT2

if not os.path.exists(CKPT_PATH):
    print('Downloading checkpoint...')
    hf_hub_download(repo_id='jkobject/scPRINT', filename='small-v2.ckpt', local_dir='.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = scPRINT2.load_from_checkpoint(CKPT_PATH, precpt_gene_emb=None, gene_pos_file=None)
if device == 'cpu':
    model = model.to(torch.float32)
model = model.to(device)
print(f'Model loaded on {device}')
print(f'Model gene count: {len(model.genes)}')

# Override to only use human (your data is human-only)
model.organisms = ['NCBITaxon:9606']

Loading model to get gene universe...
No module named 'adasplash'
FlashAttention is not installed, not using it..


/usr/local/lib/python3.12/dist-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /usr/local/lib/python3.12/dist-packages
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


small-v2.ckpt:   0%|          | 0.00/890M [00:00<?, ?B/s]

FYI: scPRINT2 is not attached to a `Trainer`.
Model loaded on cuda
Model gene count: 335182


In [ ]:
# ── Subset to model gene universe and rename to Ensembl ───────────────────────
model_gene_set = set(model.genes)
mask = [x is not None and x in model_gene_set for x in new_var_names]
print(f'Genes in model universe: {sum(mask)}')

adata_aligned = adata[:, mask].copy()
adata_aligned.var_names = [new_var_names[i] for i, m in enumerate(mask) if m]
print(f'Aligned adata: {adata_aligned.n_obs:,} cells x {adata_aligned.n_vars:,} genes')

Genes in model universe: 736
Aligned adata: 47,726 cells x 736 genes


In [ ]:
# ── Expand to full Collator gene universe (85,720 genes) ──────────────────────
# The Collator builds a boolean mask of length 85,720 (all human Ensembl genes).
# The expression matrix must match this length exactly, with zeros for unmeasured genes.
genedf = load_genes(organisms=['NCBITaxon:9606'])
full_gene_list = genedf.index.tolist()
print(f'Full Collator gene universe: {len(full_gene_list)}')

aligned_gene_set = dict(zip(adata_aligned.var_names, range(adata_aligned.n_vars)))
col_map  = [aligned_gene_set.get(g, None) for g in full_gene_list]
in_full  = [j for j, v in enumerate(col_map) if v is not None]
src_cols = [col_map[j] for j in in_full]

X_orig     = sp.csr_matrix(adata_aligned.X)
X_expanded = sp.lil_matrix((adata_aligned.n_obs, len(full_gene_list)), dtype=np.float32)
X_expanded[:, in_full] = X_orig[:, src_cols]
X_expanded = X_expanded.tocsr()

adata_full = ad.AnnData(X=X_expanded, obs=adata_aligned.obs.copy())
adata_full.var_names = full_gene_list

# Copy neighbor graph (required by SimpleAnnDataset)
adata_full.obsp['connectivities'] = adata.obsp['connectivities']
adata_full.obsp['distances']      = adata.obsp['distances']
adata_full.uns['neighbors']       = adata.uns['neighbors']

# Mark which genes are TFs (used for TF-target filtering later)
adata_full.var['isTF'] = adata_full.var_names.isin(grnutils.TF)

print(f'adata_full: {adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes')
print(f'TF count in var: {adata_full.var["isTF"].sum()}')

Full Collator gene universe: 85720
adata_full: 47,726 cells x 85,720 genes
TF count in var: 0


## Step 3: GRN Inference


In [ ]:
# =============================================================================
# scPRINT2 GRN Inference — 4 networks (CD4/CD8 × rest/act)
# Memory-optimized + GNInfer genelist bug workaround
# =============================================================================
import os, gc, torch, numpy as np, pandas as pd, scipy.sparse as sp
import anndata as ad, scanpy as sc, bionty as bt, lamindb as ln
from huggingface_hub import hf_hub_download
from scdataloader.utils import load_genes
from grnndata import utils as grnutils

torch.set_float32_matmul_precision('medium')

# ── Config ───────────────────────────────────────────────────────────────────
CKPT_PATH    = './small-v2.ckpt'
H5AD_PATH    = '/content/drive/MyDrive/benchmarking_paper/datasets/tcell_with_embeddings_filtered.h5ad'
TF_CSV_PATH  = '/content/drive/MyDrive/benchmarking_paper/datasets/humanTFs_1639_clean.csv'
GRN_OUT_DIR  = './grn_outputs'
MAX_CELLS    = 150
BATCH_SIZE   = 8
os.makedirs(GRN_OUT_DIR, exist_ok=True)

RUNS = [
    ('cd4', ['CD4_rest', 'CD4_act']),
    ('cd8', ['CD8_rest', 'CD8_act']),
]

def memreport(tag):
    import psutil
    rss = psutil.Process().memory_info().rss / 1e9
    print(f'  [mem {tag}] {rss:.1f} GB')

# ── 1. Load data ─────────────────────────────────────────────────────────────
adata = sc.read_h5ad(H5AD_PATH)
adata.obs['group4'] = (adata.obs['CD_Status'].astype(str) + '_' +
                       adata.obs['Stimulation'].astype(str))
print('Group counts:\n', adata.obs['group4'].value_counts())
memreport('after load')

# Required scPRINT metadata
for col, default in {
    'organism_ontology_term_id': 'NCBITaxon:9606',
    'assay_ontology_term_id': 'EFO:0009922',
    'self_reported_ethnicity_ontology_term_id': 'unknown',
    'disease_ontology_term_id': 'PATO:0000461',
    'sex_ontology_term_id': 'unknown',
}.items():
    if col not in adata.obs.columns:
        adata.obs[col] = default

# ── 2. Build genelist (HVGs ∪ expressed TFs in symbols) ──────────────────────
tf_df = pd.read_csv(TF_CSV_PATH)
all_tf_symbols = set(tf_df['gene_symbol'])
hvg_symbols = set(adata.var_names)        # filtered file: var_names = 2000 HVGs
expressed_tfs = all_tf_symbols & hvg_symbols
genelist_symbols = sorted(hvg_symbols | expressed_tfs)
print(f'Genelist: {len(genelist_symbols)} symbols ({len(expressed_tfs)} TFs)')

# ── 3. Symbol → Ensembl conversion ───────────────────────────────────────────
gene_df = bt.Gene.filter(organism__ontology_id='NCBITaxon:9606').to_dataframe(limit=None)
sym2ens = (gene_df.dropna(subset=['symbol', 'ensembl_gene_id'])
                  .set_index('symbol')['ensembl_gene_id'].to_dict())
ens2sym = {v: k for k, v in sym2ens.items()}
del gene_df; gc.collect()

new_var = [sym2ens.get(g) for g in adata.var_names]
genelist_ensembl = [sym2ens[s] for s in genelist_symbols if s in sym2ens]
print(f'Genelist (Ensembl): {len(genelist_ensembl)}')

# ── 4. Load model ────────────────────────────────────────────────────────────
if not os.path.exists(CKPT_PATH):
    hf_hub_download(repo_id='jkobject/scPRINT', filename='small-v2.ckpt', local_dir='.')
from scprint2 import scPRINT2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = scPRINT2.load_from_checkpoint(CKPT_PATH, precpt_gene_emb=None, gene_pos_file=None)
if device == 'cpu': model = model.to(torch.float32)
model = model.to(device)
model.organisms = ['NCBITaxon:9606']
memreport('after model load')

# ── 5. Align adata to model gene universe (NO 85k expansion) ─────────────────
model_genes = set(model.genes)
mask = np.array([x is not None and x in model_genes for x in new_var])
print(f'Genes in model universe: {mask.sum()}')

adata_aligned = adata[:, mask].copy()
adata_aligned.var_names = [new_var[i] for i, m in enumerate(mask) if m]
adata_aligned.var['symbol'] = [ens2sym.get(g, g) for g in adata_aligned.var_names]
adata_aligned.var['isTF'] = adata_aligned.var['symbol'].isin(grnutils.TF)

if not sp.issparse(adata_aligned.X):
    adata_aligned.X = sp.csr_matrix(adata_aligned.X)

# Free the original full adata
del adata, new_var, mask
gc.collect()
memreport('after alignment')
print(f'adata_aligned: {adata_aligned.shape}, '
      f'TFs: {int(adata_aligned.var.isTF.sum())}')

# ── 6. GRN inference: monkey-patch + 4 networks ──────────────────────────────
from scprint2.tasks import GNInfer

# Bug workaround: scprint2's GNInfer.__init__ accesses self.genelist before
# assigning it. Pre-set it so the buggy line doesn't crash.
_original_init = GNInfer.__init__
def _patched_init(self, *args, **kwargs):
    self.genelist = kwargs.get('genelist', []) or []
    _original_init(self, *args, **kwargs)
GNInfer.__init__ = _patched_init

grn_inferer = GNInfer(
    how='some',
    genelist=genelist_ensembl,
    preprocess='softmax',
    head_agg='mean',
    filtration='none',
    max_cells=MAX_CELLS,
    drop_unexpressed=True,
    batch_size=BATCH_SIZE,
    cell_type_col='group4',
    layer=list(range(model.nlayers)),
)

# Filter low-quality cells once
counts = np.asarray(adata_aligned.X.sum(1)).ravel()
keep = counts > 500
print(f'Cells passing >500 counts: {keep.sum():,} / {adata_aligned.n_obs:,}')

for lineage, states in RUNS:
    state_mask = adata_aligned.obs['group4'].isin(states).values & keep
    sub = adata_aligned[state_mask].copy()
    print(f'\n=== {lineage.upper()}: {sub.n_obs} cells across {states} ===')
    memreport(f'{lineage} sub created')

    for state in states:
        n = (sub.obs['group4'] == state).sum()
        print(f'  {state}: {n} cells')
        if n < 20:
            print(f'  Skipping {state} (too few cells)'); continue

        grn = grn_inferer(model, sub, cell_type=state)
        grn.var['symbol'] = [ens2sym.get(g, g) for g in grn.var_names]
        grn.var['isTF']   = grn.var['symbol'].isin(grnutils.TF)

        out_path = os.path.join(GRN_OUT_DIR, f'{state}_grn.h5ad')
        grn.write_h5ad(out_path)
        print(f'  → {out_path} | shape {grn.varp["GRN"].shape} | '
              f'{int(grn.var.isTF.sum())} TFs')

        del grn
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()
        memreport(f'  after {state}')

    del sub
    gc.collect()

print(f'\nDone. Files in {GRN_OUT_DIR}/')

Group counts:
 group4
CD4_act     18144
CD4_rest    16808
CD8_rest     6669
CD8_act      6105
Name: count, dtype: int64
  [mem after load] 8.2 GB
Genelist: 2000 symbols (68 TFs)
Genelist (Ensembl): 1191
FYI: scPRINT2 is not attached to a `Trainer`.
  [mem after model load] 8.0 GB
Genes in model universe: 736
  [mem after alignment] 7.7 GB
adata_aligned: (47726, 736), TFs: 68
Cells passing >500 counts: 6 / 47,726

=== CD4: 3 cells across ['CD4_rest', 'CD4_act'] ===
  [mem cd4 sub created] 7.7 GB
  CD4_rest: 1 cells
  Skipping CD4_rest (too few cells)
  CD4_act: 2 cells
  Skipping CD4_act (too few cells)

=== CD8: 3 cells across ['CD8_rest', 'CD8_act'] ===
  [mem cd8 sub created] 7.7 GB
  CD8_rest: 1 cells
  Skipping CD8_rest (too few cells)
  CD8_act: 2 cells
  Skipping CD8_act (too few cells)

Done. Files in ./grn_outputs/


In [ ]:
adata

AnnData object with n_obs × n_vars = 47726 × 2000
    obs: 'CD_Status', 'Stimulation', 'nUMI', 'leiden'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'tsne', 'umap'
    obsm: 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [ ]:
print(adata.obs.columns.tolist())
print(adata.obs.head())

In [ ]:
grn = grn_inferer(model, adata_expressed, cell_type=cell_type)
print("grn.var columns:", grn.var.columns.tolist())
print("grn.var.index[:5]:", grn.var.index[:5].tolist())

In [ ]:
import bionty as bt

# Build ensembl -> symbol mapping
gene_df = bt.Gene.filter(organism__ontology_id='NCBITaxon:9606').to_dataframe(limit=None)
ensembl_to_symbol = (
    gene_df.dropna(subset=['ensembl_gene_id', 'symbol'])
    .set_index('ensembl_gene_id')['symbol']
    .to_dict()
)

# Add symbol column to adata_full.var (var_names are Ensembl IDs)
adata_full.var['symbol'] = [ensembl_to_symbol.get(g, g) for g in adata_full.var_names]

# Verify
print(adata_full.var[['symbol']].head(10))
print(f"Non-null symbols: {adata_full.var['symbol'].notna().sum()} / {adata_full.n_vars}")

In [ ]:
# ── Run GRN inference for each cell group ─────────────────────────────────────
adata_expressed = adata_full[adata_full.X.sum(1) > 500]
print(f'Cells with >500 counts: {adata_expressed.n_obs:,}')

cell_types = adata_full.obs[CELL_TYPE_COL].dropna().unique().tolist()
print(f'Cell groups to process: {cell_types}')

grns = {}
for cell_type in cell_types:
    print(f'\nInferring GRN for: {cell_type}')
    n_cells = (adata_expressed.obs[CELL_TYPE_COL] == cell_type).sum()
    print(f'  Available cells: {n_cells}')

    if n_cells < 20:
        print(f'  Skipping (too few cells)')
        continue

    grn = grn_inferer(model, adata_expressed, cell_type=cell_type)

    # Aggregate per-head/layer tensor to single 2D matrix
    grn.varp['GRN_all_heads'] = grn.varp['GRN'].copy()  # (n_genes, n_genes, n_layers*n_heads)
    grn.varp['GRN']           = grn.varp['GRN'].mean(-1) # (n_genes, n_genes)

    grns[cell_type] = grn

    # Fix index/column name clash before writing
    if 'symbol' in grn.var.columns and grn.var.index.name == 'symbol':
        grn.var = grn.var.rename(columns={'symbol': 'gene_symbol'})

    out_path = os.path.join(GRN_OUT_DIR, cell_type.replace(' ', '_') + 'v2_grn.h5ad')
    grn.write_h5ad(out_path)
    print(f'  Saved to: {out_path}')
    print(f'  GRN shape: {grn.varp["GRN"].shape}')

print(f'\nDone. GRNs computed for: {list(grns.keys())}')

## Step 4: Quick Inspection

Look at the top regulatory edges and TF enrichment for each cell group.

In [ ]:
import pandas as pd

for cell_type, grn in grns.items():
    print(f'\n=== {cell_type} ===')
    print(f'Genes in network: {grn.n_vars}')
    print(f'TFs in network:   {grn.var["isTF"].sum() if "isTF" in grn.var.columns else "N/A"}')

    # Top edges by attention weight
    G = grn.varp['GRN']
    genes = grn.var_names.tolist()
    rows, cols = np.unravel_index(np.argsort(G.ravel())[-20:], G.shape)
    top_edges = pd.DataFrame({
        'regulator': [genes[r] for r in rows],
        'target':    [genes[c] for c in cols],
        'weight':    [G[r, c] for r, c in zip(rows, cols)]
    }).sort_values('weight', ascending=False)

    print('Top 10 edges:')
    print(top_edges.head(10).to_string(index=False))

In [ ]:
import numpy as np

for cell_type, grn in grns.items():
    G = grn.varp['GRN'].copy()

    # Remove self-loops
    np.fill_diagonal(G, 0)
    grn.varp['GRN_no_diag'] = G

    # Fix TF labeling — grnutils.TF uses symbols, map via gene_symbol column
    symbol_col = 'gene_symbol' if 'gene_symbol' in grn.var.columns else 'symbol'
    if symbol_col in grn.var.columns:
        grn.var['isTF'] = grn.var[symbol_col].isin(grnutils.TF)

    genes = grn.var_names.tolist()
    rows, cols = np.unravel_index(np.argsort(G.ravel())[-10:], G.shape)
    print(f'\n=== {cell_type} (no diagonal) ===')
    print(f'TFs in network: {grn.var["isTF"].sum()}')
    for r, c in zip(rows[::-1], cols[::-1]):
        reg_sym = grn.var[symbol_col].iloc[r] if symbol_col in grn.var.columns else genes[r]
        tgt_sym = grn.var[symbol_col].iloc[c] if symbol_col in grn.var.columns else genes[c]
        print(f'  {reg_sym:12s} → {tgt_sym:12s}  {G[r,c]:.4f}')

In [ ]:
## FIRST VERSION

"""
=============================================================================
GRN Analysis  O

— rest vs act T cell Gene Regulatory Networks
=============================================================================
Loads the two GRNs output by scPRINT2 GNInfer and derives biological insights
by comparing regulatory structure between resting and activated T cells.

Sections:
  1. Load & clean           — remove self-loops, fix TF labels
  2. Network statistics     — degree, density, hub genes
  3. TF analysis            — which TFs are most regulatory active
  4. Differential edges     — which connections change rest -> act
  5. Top rewired genes      — genes that gain/lose the most regulation
  6. Hub gene comparison    — most influential genes in each state
  7. Visualisation          — plots saved to grn_analysis/
  8. Save summary tables    — CSVs for downstream use
=============================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scanpy as sc
from anndata.utils import make_index_unique

try:
    from grnndata import utils as grnutils
    TF_SET = set(grnutils.TF)
except Exception:
    print("Warning: grnndata not available — TF labels will be skipped")
    TF_SET = set()

# ── Config ────────────────────────────────────────────────────────────────────
GRN_OUT_DIR = "./grn_outputs"
SAVE_DIR    = "./grn_analysis"
os.makedirs(SAVE_DIR, exist_ok=True)

TOP_N        = 20
DIFF_THRESH  = 0.01


# =============================================================================
# 1. LOAD & CLEAN
# =============================================================================

print("\n" + "="*70)
print("SECTION 1: Loading GRNs and cleaning")
print("="*70)
print("Reading the two GRN h5ad files produced by scPRINT2.")
print("Each file contains a 2000x2000 attention-weight matrix where")
print("entry [i,j] = how strongly gene i regulates gene j in that cell state.")
print("Self-loops (a gene attending to itself) are removed as they are")
print("an artifact of the attention mechanism, not real regulation.\n")

grn_rest = sc.read_h5ad(os.path.join(GRN_OUT_DIR, "rest_grn.h5ad"))
grn_act  = sc.read_h5ad(os.path.join(GRN_OUT_DIR, "act_grn.h5ad"))

def clean_grn(grn, name):
    G = grn.varp["GRN"].copy().astype(np.float32)
    np.fill_diagonal(G, 0)

    sym_col = "gene_symbol" if "gene_symbol" in grn.var.columns else \
              "symbol"      if "symbol"      in grn.var.columns else None

    if sym_col:
        grn.var["isTF"]        = grn.var[sym_col].isin(TF_SET)
        grn.var["symbol_clean"] = grn.var[sym_col].astype(str)
    else:
        grn.var["isTF"]        = False
        grn.var["symbol_clean"] = grn.var_names.astype(str)

    n_tf = int(grn.var["isTF"].sum())
    print(f"  {name}: {grn.n_vars} genes, {n_tf} identified as TFs, self-loops removed.")
    return G

print("Cleaning rest GRN...")
G_rest = clean_grn(grn_rest, "rest")
print("\nCleaning act GRN...")
G_act  = clean_grn(grn_act, "act")

rest_genes = grn_rest.var_names.tolist()
act_genes  = grn_act.var_names.tolist()
shared     = list(set(rest_genes) & set(act_genes))

print(f"\nGenes in rest network : {len(rest_genes)}")
print(f"Genes in act network  : {len(act_genes)}")
print(f"Genes shared in both  : {len(shared)}")
print("Only shared genes are used for the differential analysis so we")
print("are comparing the same set of genes across both conditions.")

rest_idx = [rest_genes.index(g) for g in shared]
act_idx  = [act_genes.index(g)  for g in shared]
G_rest_shared = G_rest[np.ix_(rest_idx, rest_idx)]
G_act_shared  = G_act[np.ix_(act_idx,  act_idx)]
shared_symbols = grn_rest.var["symbol_clean"].iloc[rest_idx].tolist()


# =============================================================================
# 2. NETWORK STATISTICS
# =============================================================================

print("\n" + "="*70)
print("SECTION 2: Network-level statistics")
print("="*70)
print("These metrics describe the overall structure of each regulatory network.")
print("Out-degree = number of other genes a gene regulates (how active a regulator it is).")
print("In-degree  = number of regulators pointing at a gene (how heavily regulated it is).")
print("Density    = fraction of all possible regulatory edges that are actually present.\n")

def network_stats(G, name):
    n            = G.shape[0]
    total_weight = float(G.sum())
    nonzero      = G[G > 0]
    mean_weight  = float(nonzero.mean()) if len(nonzero) > 0 else 0
    out_degree   = (G > 0).sum(axis=1)
    in_degree    = (G > 0).sum(axis=0)
    density      = float((G > 0).sum()) / (n * (n - 1))

    print(f"  [{name}]")
    print(f"    Total regulatory weight  : {total_weight:.2f}")
    print(f"      -> Higher total weight means more overall regulatory activity")
    print(f"    Mean nonzero edge weight : {mean_weight:.5f}")
    print(f"      -> Average strength of edges that exist")
    print(f"    Network density          : {density*100:.2f}% of all possible edges are active")
    print(f"    Max out-degree           : {out_degree.max()} — one gene regulates up to this many others")
    print(f"    Max in-degree            : {in_degree.max()} — one gene receives input from up to this many regulators")
    return out_degree, in_degree

out_rest, in_rest = network_stats(G_rest, "rest")
print()
out_act,  in_act  = network_stats(G_act,  "act")


# =============================================================================
# 3. TF ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("SECTION 3: Transcription factor regulatory activity")
print("="*70)
print("We sum the outgoing edge weights for each TF to get a regulatory score.")
print("A higher score means the TF drives more genes more strongly.")
print("Comparing scores between rest and act shows which TFs become")
print("more or less active during T cell activation.\n")

def tf_scores(G, grn, name):
    is_tf  = grn.var["isTF"].values
    syms   = grn.var["symbol_clean"].values
    scores = G.sum(axis=1)

    tf_df = pd.DataFrame({
        "symbol": syms[is_tf],
        "regulatory_score": scores[is_tf],
    }).sort_values("regulatory_score", ascending=False).reset_index(drop=True)

    print(f"  [{name}] Top {min(TOP_N, len(tf_df))} most active TFs by total outgoing weight:")
    if len(tf_df) == 0:
        print("    No TFs identified — check that gene_symbol column maps to known TF names.")
    else:
        print(tf_df.head(TOP_N).to_string(index=False))
    return tf_df

tf_rest = tf_scores(G_rest, grn_rest, "rest")
print()
tf_act  = tf_scores(G_act,  grn_act,  "act")


# =============================================================================
# 4. DIFFERENTIAL EDGES
# =============================================================================

print("\n" + "="*70)
print("SECTION 4: Differential regulatory edges (rest vs act)")
print("="*70)
print("We compute (act weight - rest weight) for every gene pair.")
print("Positive delta = a regulatory link that is NEW or STRONGER in activated cells.")
print("Negative delta = a regulatory link that DISAPPEARS or WEAKENS upon activation.")
print("These are the specific gene-gene regulatory relationships that change")
print("when a T cell transitions from resting to activated state.")
print(f"We only report edges where |delta| > {DIFF_THRESH} to focus on meaningful changes.\n")

G_diff = G_act_shared - G_rest_shared

n_shared = len(shared)
rows, cols = np.meshgrid(range(n_shared), range(n_shared), indexing="ij")
rows = rows.ravel(); cols = cols.ravel()
delta = G_diff.ravel()

mask = np.abs(delta) > DIFF_THRESH
diff_df = pd.DataFrame({
    "regulator": [shared_symbols[r] for r in rows[mask]],
    "target":    [shared_symbols[c] for c in cols[mask]],
    "delta":     delta[mask],
    "direction": ["gained" if d > 0 else "lost" for d in delta[mask]],
}).sort_values("delta", key=abs, ascending=False).reset_index(drop=True)

print(f"  Total rewired edges : {len(diff_df)}")
print(f"  Gained in activation: {(diff_df.direction == 'gained').sum()}")
print(f"    -> These are regulatory links that form when T cells are stimulated")
print(f"  Lost in activation  : {(diff_df.direction == 'lost').sum()}")
print(f"    -> These are regulatory links that dissolve upon stimulation")

print(f"\n  Top {TOP_N} GAINED edges — new regulatory links in activated T cells:")
print(diff_df[diff_df.direction == "gained"].head(TOP_N).to_string(index=False))

print(f"\n  Top {TOP_N} LOST edges — regulatory links that dissolve upon activation:")
print(diff_df[diff_df.direction == "lost"].head(TOP_N).to_string(index=False))


# =============================================================================
# 5. TOP REWIRED GENES
# =============================================================================

print("\n" + "="*70)
print("SECTION 5: Most rewired genes")
print("="*70)
print("We measure how much each gene's total incoming and outgoing regulatory")
print("weight changes between rest and act.")
print("out_delta > 0 means the gene REGULATES MORE genes in the activated state.")
print("in_delta  > 0 means the gene IS MORE HEAVILY REGULATED in the activated state.")
print("Genes with large changes in either direction are key players in the")
print("transcriptional reprogramming that occurs during T cell activation.\n")

out_delta = G_act_shared.sum(axis=1) - G_rest_shared.sum(axis=1)
in_delta  = G_act_shared.sum(axis=0) - G_rest_shared.sum(axis=0)

rewired_df = pd.DataFrame({
    "symbol":          shared_symbols,
    "out_delta":       out_delta,
    "in_delta":        in_delta,
    "abs_total_delta": np.abs(out_delta) + np.abs(in_delta),
}).sort_values("abs_total_delta", ascending=False).reset_index(drop=True)

print(f"  Top {TOP_N} most rewired genes overall:")
print(rewired_df.head(TOP_N).to_string(index=False))

print(f"\n  Genes that become STRONGER REGULATORS upon activation (top {TOP_N}):")
print(f"  -> These likely include key TFs driving the activation program")
print(rewired_df.nlargest(TOP_N, "out_delta")[["symbol","out_delta"]].to_string(index=False))

print(f"\n  Genes that become MORE HEAVILY REGULATED upon activation (top {TOP_N}):")
print(f"  -> These are downstream targets that get switched on during activation")
print(rewired_df.nlargest(TOP_N, "in_delta")[["symbol","in_delta"]].to_string(index=False))


# =============================================================================
# 6. HUB GENE COMPARISON
# =============================================================================

print("\n" + "="*70)
print("SECTION 6: Hub gene comparison")
print("="*70)
print("Hub genes have the highest total outgoing regulatory weight —")
print("they are the master regulators of each cell state.")
print("A gene that is a hub in act but not rest is likely driving activation.")
print("A gene that is a hub in rest but not act is likely a homeostasis maintainer.\n")

def hub_genes(G, symbols, name, n=TOP_N):
    scores  = G.sum(axis=1)
    top_idx = np.argsort(scores)[::-1][:n]
    df = pd.DataFrame({
        "symbol": [symbols[i] for i in top_idx],
        "total_outgoing_weight": scores[top_idx],
    })
    print(f"  [{name}] Top {n} hub regulators:")
    print(df.to_string(index=False))
    return set(df["symbol"].tolist())

hubs_rest = hub_genes(G_rest_shared, shared_symbols, "rest")
print()
hubs_act  = hub_genes(G_act_shared,  shared_symbols, "act")

activation_specific = hubs_act - hubs_rest
rest_specific       = hubs_rest - hubs_act
shared_hubs         = hubs_rest & hubs_act

print(f"\n  Hubs in BOTH states (stable core regulators): {sorted(shared_hubs)}")
print(f"    -> These genes maintain regulatory control regardless of activation state")
print(f"  Hubs GAINED in activation (activation drivers): {sorted(activation_specific)}")
print(f"    -> These genes become dominant regulators specifically when T cells are stimulated")
print(f"  Hubs LOST in activation (rest-specific regulators): {sorted(rest_specific)}")
print(f"    -> These genes regulate the resting state and are suppressed upon activation")


# =============================================================================
# 7. VISUALISATION
# =============================================================================

print("\n" + "="*70)
print("SECTION 7: Generating plots")
print("="*70)

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(G_rest[G_rest > 0].ravel(), bins=80, alpha=0.6,
         color="steelblue", density=True, label="rest")
ax1.hist(G_act[G_act > 0].ravel(),  bins=80, alpha=0.6,
         color="darkorange", density=True, label="act")
ax1.set_title("Edge weight distribution\n(nonzero edges only)", fontsize=11)
ax1.set_xlabel("Attention weight"); ax1.set_ylabel("Density")
ax1.legend()
ax1.text(0.02, 0.95,
         "Compares whether one state has\ngenerally stronger regulatory links.",
         transform=ax1.transAxes, fontsize=8, va="top", color="gray")

ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(G_diff.ravel(), bins=100, color="mediumpurple", alpha=0.8)
ax2.axvline(0, color="black", linewidth=1)
ax2.axvline( DIFF_THRESH, color="red", linewidth=1, linestyle="--",
             label=f"threshold ±{DIFF_THRESH}")
ax2.axvline(-DIFF_THRESH, color="red", linewidth=1, linestyle="--")
ax2.set_title("Edge weight changes\n(act - rest)", fontsize=11)
ax2.set_xlabel("Delta weight"); ax2.set_ylabel("Count")
ax2.legend(fontsize=8)
ax2.text(0.02, 0.95,
         "Right of 0 = edges gained in activation.\nLeft = edges lost.",
         transform=ax2.transAxes, fontsize=8, va="top", color="gray")

ax3 = fig.add_subplot(gs[0, 2])
top_rw = rewired_df.head(15)
colors = ["darkorange" if d > 0 else "steelblue" for d in top_rw["out_delta"]]
ax3.barh(top_rw["symbol"][::-1], top_rw["out_delta"][::-1], color=colors[::-1])
ax3.axvline(0, color="black", linewidth=0.8)
ax3.set_title("Top 15 rewired regulators\n(outgoing weight delta)", fontsize=11)
ax3.set_xlabel("Change in outgoing weight (act - rest)")
ax3.text(0.02, 0.02,
         "Orange = more active in act.\nBlue = more active in rest.",
         transform=ax3.transAxes, fontsize=8, va="bottom", color="gray")

ax4 = fig.add_subplot(gs[1, 0])
all_hubs   = sorted(hubs_rest | hubs_act)
rest_sc    = {s: G_rest_shared.sum(axis=1)[shared_symbols.index(s)]
              for s in all_hubs if s in shared_symbols}
act_sc     = {s: G_act_shared.sum(axis=1)[shared_symbols.index(s)]
              for s in all_hubs if s in shared_symbols}
hub_df     = pd.DataFrame({"rest": rest_sc, "act": act_sc}).fillna(0)
x = np.arange(len(hub_df)); w = 0.35
ax4.bar(x - w/2, hub_df["rest"], w, label="rest",  color="steelblue",  alpha=0.8)
ax4.bar(x + w/2, hub_df["act"],  w, label="act",   color="darkorange", alpha=0.8)
ax4.set_xticks(x)
ax4.set_xticklabels(hub_df.index, rotation=45, ha="right", fontsize=8)
ax4.set_title("Hub gene regulatory scores\n(rest vs act)", fontsize=11)
ax4.set_ylabel("Total outgoing weight")
ax4.legend()
ax4.text(0.02, 0.95,
         "Taller bar = stronger regulatory\ninfluence in that state.",
         transform=ax4.transAxes, fontsize=8, va="top", color="gray")

ax5 = fig.add_subplot(gs[1, 1])
top_gained = diff_df[diff_df.direction == "gained"].head(15)
if len(top_gained) > 0:
    pivot_g = top_gained.pivot_table(
        index="regulator", columns="target", values="delta", fill_value=0)
    sns.heatmap(pivot_g, ax=ax5, cmap="Oranges", cbar_kws={"shrink": 0.6})
    ax5.set_title("Top gained edges (act - rest)\nRegulator -> Target", fontsize=11)
    ax5.tick_params(axis="both", labelsize=7)
else:
    ax5.text(0.5, 0.5, "No gained edges\nabove threshold",
             ha="center", va="center", transform=ax5.transAxes)
    ax5.set_title("Top gained edges", fontsize=11)
ax5.text(0.02, -0.28,
         "New/strengthened regulatory links\nin activated T cells.",
         transform=ax5.transAxes, fontsize=8, color="gray")

ax6 = fig.add_subplot(gs[1, 2])
top_lost = diff_df[diff_df.direction == "lost"].head(15)
if len(top_lost) > 0:
    pivot_l = top_lost.pivot_table(
        index="regulator", columns="target", values="delta", fill_value=0)
    sns.heatmap(pivot_l.abs(), ax=ax6, cmap="Blues", cbar_kws={"shrink": 0.6})
    ax6.set_title("Top lost edges (rest - act)\nRegulator -> Target", fontsize=11)
    ax6.tick_params(axis="both", labelsize=7)
else:
    ax6.text(0.5, 0.5, "No lost edges\nabove threshold",
             ha="center", va="center", transform=ax6.transAxes)
    ax6.set_title("Top lost edges", fontsize=11)
ax6.text(0.02, -0.28,
         "Regulatory links that dissolve\nwhen T cells are activated.",
         transform=ax6.transAxes, fontsize=8, color="gray")

fig.suptitle("scPRINT2 GRN Analysis: Resting vs Activated T Cells",
             fontsize=15, fontweight="bold", y=1.01)

plot_path = os.path.join(SAVE_DIR, "grn_analysis.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"  Plot saved to: {plot_path}")


# =============================================================================
# 8. SAVE SUMMARY TABLES
# =============================================================================

print("\n" + "="*70)
print("SECTION 8: Saving summary tables")
print("="*70)

diff_path    = os.path.join(SAVE_DIR, "differential_edges.csv")
rewired_path = os.path.join(SAVE_DIR, "rewired_genes.csv")
tf_path      = os.path.join(SAVE_DIR, "tf_scores.csv")

diff_df.to_csv(diff_path, index=False)
rewired_df.to_csv(rewired_path, index=False)

tf_combined = (
    tf_rest.rename(columns={"regulatory_score": "score_rest"})
    .merge(tf_act.rename(columns={"regulatory_score": "score_act"}),
           on="symbol", how="outer")
    .fillna(0)
)
tf_combined["delta"] = tf_combined["score_act"] - tf_combined["score_rest"]
tf_combined = tf_combined.sort_values("delta", key=abs, ascending=False)
tf_combined.to_csv(tf_path, index=False)

print(f"  differential_edges.csv -> all rewired edges with direction and delta weight")
print(f"  rewired_genes.csv      -> per-gene summary of how much regulation changed")
print(f"  tf_scores.csv          -> TF regulatory scores in rest and act with delta")
print(f"\n  Files saved to: {SAVE_DIR}/")

print("\n" + "="*70)
print("DONE — Key findings summary")
print("="*70)
print(f"  Shared genes analysed         : {len(shared)}")
print(f"  Total rewired edges           : {len(diff_df)}")
print(f"  Edges gained in activation    : {(diff_df.direction == 'gained').sum()}")
print(f"  Edges lost in activation      : {(diff_df.direction == 'lost').sum()}")
print(f"  Activation-specific hub genes : {sorted(activation_specific)}")
print(f"  Stable hub genes (both states): {sorted(shared_hubs)}")
print(f"  Hub genes lost in activation  : {sorted(rest_specific)}")
print("\nRecommended next step: cross-reference activation-specific hub genes")
print("against known T cell activation TFs (NFAT, AP-1, NF-kB) to validate")
print("whether scPRINT recovers known biology from attention weights alone.")b

In [ ]:
print("rest genes sample:", grn_rest.var_names[:5].tolist())
print("act genes sample:", grn_act.var_names[:5].tolist())
print("shared count:", len(shared))
print("G_rest_shared shape:", G_rest_shared.shape)
print("G_act_shared shape:", G_act_shared.shape)

In [ ]:
rest_set = set(grn_rest.var_names.tolist())
act_set  = set(grn_act.var_names.tolist())

print(f"rest unique genes: {len(rest_set)}")
print(f"act unique genes:  {len(act_set)}")
print(f"overlap:           {len(rest_set & act_set)}")

# Check if there are duplicates within each
print(f"\nDuplicates in rest: {len(grn_rest.var_names) - len(rest_set)}")
print(f"Duplicates in act:  {len(grn_act.var_names)  - len(act_set)}")

# Sample from each to visually compare
print("\nFirst 5 rest:", sorted(rest_set)[:5])
print("First 5 act: ", sorted(act_set)[:5])

In [ ]:
import shutil
import os

# Mount Drive if not already mounted
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DEST = '/content/drive/MyDrive/tcell_grn_results'
os.makedirs(DRIVE_DEST, exist_ok=True)

for folder in ['grn_outputs', 'grn_analysis']:
    src = f'./{folder}'
    dst = os.path.join(DRIVE_DEST, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = os.listdir(dst)
        print(f'Copied {folder}/ -> {dst}')
        print(f'  Files: {files}')
    else:
        print(f'Warning: {src} not found, skipping')

print(f'\nDone. Everything saved to: {DRIVE_DEST}')

In [ ]:
# ── TF-only network: zero out non-TF regulators ───────────────────────────────
# This is the standard approach from bench_omni: keep only edges where
# the regulator is a known transcription factor
for cell_type, grn in grns.items():
    if 'isTF' in grn.var.columns:
        G_tf = grn.varp['GRN'].copy()
        G_tf[~grn.var['isTF'].values, :] = 0
        grn.varp['GRN_tf_only'] = G_tf
        print(f'{cell_type}: TF-only GRN stored in varp["GRN_tf_only"]')

In [ ]:
# ── Optional: benchmark against OmniPath using BenGRN ─────────────────────────
# Uncomment if you want a quantitative benchmark
# from bengrn import BenGRN
# metrics = {}
# for cell_type, grn in grns.items():
#     metrics[cell_type + '_mean']    = BenGRN(grn).scprint_benchmark()
#     grn.varp['GRN'] = grn.varp['GRN_tf_only']
#     metrics[cell_type + '_tf_mean'] = BenGRN(grn).scprint_benchmark()
# print(metrics)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

# ── Load embeddings ───────────────────────────────────────────────────────────
EMBEDDING_PATH = '/content/drive/MyDrive/tcell_embedded.h5ad'  # update if needed
adata = sc.read_h5ad(EMBEDDING_PATH)
EMB_KEY = 'scprint_emb_cell_type_ontology_term_id'
emb = adata.obsm[EMB_KEY]
print(adata)
print('\nobsm keys:', list(adata.obsm.keys()))
print(f'Embedding shape: {emb.shape}')

# ── 1. UMAP ───────────────────────────────────────────────────────────────────
sc.pp.neighbors(adata, use_rep=EMB_KEY, n_neighbors=15)
sc.tl.umap(adata, random_state=42)
sc.pl.umap(adata, color=['stimulation', 'cd_status', 'conv_pred_cell_type_ontology_term_id'],
           ncols=3, frameon=False, size=3)

# ── 2. Predictions vs ground truth ───────────────────────────────────────────
for col in ['stimulation', 'cd_status']:
    print(f'\n=== Predicted cell type by {col} ===')
    ct = (adata.obs.groupby(col)['conv_pred_cell_type_ontology_term_id']
          .value_counts(normalize=True).mul(100).round(1).rename('pct').reset_index())
    for grp, sub in ct.groupby(col):
        print(f'\n-- {grp} --')
        print(sub.head(10).to_string(index=False))

# ── 3. Silhouette score ───────────────────────────────────────────────────────
idx = np.random.choice(adata.n_obs, size=min(5000, adata.n_obs), replace=False)
emb_sub = emb[idx]
for col in ['stimulation', 'cd_status']:
    labels = adata.obs[col].values[idx]
    score = silhouette_score(emb_sub, labels)
    print(f'Silhouette ({col}): {score:.4f}')

# ── 4. Differential embedding (rest vs act) ───────────────────────────────────
emb_df = pd.DataFrame(emb, index=adata.obs_names)
emb_df['stimulation'] = adata.obs['stimulation'].values
diff = emb_df.groupby('stimulation').mean().diff().iloc[-1].abs().sort_values(ascending=False)

top_dims = diff.head(20).index.tolist()
fig, axes = plt.subplots(4, 5, figsize=(18, 12))
axes = axes.flatten()
for i, dim in enumerate(top_dims):
    for grp, color in [('rest', '#4C72B0'), ('act', '#DD8452')]:
        vals = emb_df[emb_df['stimulation'] == grp][dim]
        axes[i].hist(vals, bins=40, alpha=0.6, color=color, label=grp, density=True)
    axes[i].set_title(f'Dim {dim}', fontsize=9)
    if i == 0:
        axes[i].legend(fontsize=8)
plt.suptitle('Top 20 differential embedding dimensions: rest vs act', fontsize=13)
plt.tight_layout()
plt.show()

# ── 5. Louvain clustering ─────────────────────────────────────────────────────
sc.tl.louvain(adata, resolution=0.5, key_added='louvain_emb')
sc.pl.umap(adata, color=['louvain_emb', 'stimulation', 'cd_status'], ncols=3, frameon=False, size=3)
purity = (adata.obs.groupby('louvain_emb')['stimulation']
          .value_counts(normalize=True).unstack(fill_value=0))
print('\nCluster purity (stimulation):')
print(purity.round(2).to_string())

In [ ]:
import scipy.sparse as sp

rest_var = np.asarray(sp.csr_matrix(rest_cells.X).power(2).mean(axis=0) -
                      np.power(sp.csr_matrix(rest_cells.X).mean(axis=0), 2)).ravel()
act_var  = np.asarray(sp.csr_matrix(act_cells.X).power(2).mean(axis=0) -
                      np.power(sp.csr_matrix(act_cells.X).mean(axis=0), 2)).ravel()

rest_top2000 = set(adata_full.var_names[np.argsort(rest_var)[::-1][:2000]])
act_top2000  = set(adata_full.var_names[np.argsort(act_var)[::-1][:2000]])

print(f"Top 2000 variable genes in rest: {len(rest_top2000)}")
print(f"Top 2000 variable genes in act:  {len(act_top2000)}")
print(f"Overlap in top 2000:             {len(rest_top2000 & act_top2000)}")

grn_rest_genes = set(grn_rest.var_names.tolist())
grn_act_genes  = set(grn_act.var_names.tolist())

print(f"\nOverlap: GNInfer rest vs our computed rest top2000: {len(grn_rest_genes & rest_top2000)}")
print(f"Overlap: GNInfer act  vs our computed act  top2000: {len(grn_act_genes  & act_top2000)}")
print(f"Overlap: GNInfer rest vs our computed act  top2000: {len(grn_rest_genes & act_top2000)}")
print(f"Overlap: GNInfer act  vs our computed rest top2000: {len(grn_act_genes  & rest_top2000)}")

In [ ]:
## FIRST VERSION

"""
=============================================================================
GRN Analysis  O

— rest vs act T cell Gene Regulatory Networks
=============================================================================
Loads the two GRNs output by scPRINT2 GNInfer and derives biological insights
by comparing regulatory structure between resting and activated T cells.

Sections:
  1. Load & clean           — remove self-loops, fix TF labels
  2. Network statistics     — degree, density, hub genes
  3. TF analysis            — which TFs are most regulatory active
  4. Differential edges     — which connections change rest -> act
  5. Top rewired genes      — genes that gain/lose the most regulation
  6. Hub gene comparison    — most influential genes in each state
  7. Visualisation          — plots saved to grn_analysis/
  8. Save summary tables    — CSVs for downstream use
=============================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scanpy as sc
from anndata.utils import make_index_unique

try:
    from grnndata import utils as grnutils
    TF_SET = set(grnutils.TF)
except Exception:
    print("Warning: grnndata not available — TF labels will be skipped")
    TF_SET = set()

# ── Config ────────────────────────────────────────────────────────────────────
GRN_OUT_DIR = "./grn_outputs"
SAVE_DIR    = "./grn_analysis"
os.makedirs(SAVE_DIR, exist_ok=True)

TOP_N        = 20
DIFF_THRESH  = 0.01


# =============================================================================
# 1. LOAD & CLEAN
# =============================================================================

print("\n" + "="*70)
print("SECTION 1: Loading GRNs and cleaning")
print("="*70)
print("Reading the two GRN h5ad files produced by scPRINT2.")
print("Each file contains a 2000x2000 attention-weight matrix where")
print("entry [i,j] = how strongly gene i regulates gene j in that cell state.")
print("Self-loops (a gene attending to itself) are removed as they are")
print("an artifact of the attention mechanism, not real regulation.\n")

grn_rest = sc.read_h5ad(os.path.join(GRN_OUT_DIR, "restv2_grn.h5ad"))
grn_act  = sc.read_h5ad(os.path.join(GRN_OUT_DIR, "actv2_grn.h5ad"))

def clean_grn(grn, name):
    G = grn.varp["GRN"].copy().astype(np.float32)
    np.fill_diagonal(G, 0)

    sym_col = "gene_symbol" if "gene_symbol" in grn.var.columns else \
              "symbol"      if "symbol"      in grn.var.columns else None

    if sym_col:
        grn.var["isTF"]        = grn.var[sym_col].isin(TF_SET)
        grn.var["symbol_clean"] = grn.var[sym_col].astype(str)
    else:
        grn.var["isTF"]        = False
        grn.var["symbol_clean"] = grn.var_names.astype(str)

    n_tf = int(grn.var["isTF"].sum())
    print(f"  {name}: {grn.n_vars} genes, {n_tf} identified as TFs, self-loops removed.")
    return G

print("Cleaning rest GRN...")
G_rest = clean_grn(grn_rest, "rest")
print("\nCleaning act GRN...")
G_act  = clean_grn(grn_act, "act")

rest_genes = grn_rest.var_names.tolist()
act_genes  = grn_act.var_names.tolist()
shared     = list(set(rest_genes) & set(act_genes))

print(f"\nGenes in rest network : {len(rest_genes)}")
print(f"Genes in act network  : {len(act_genes)}")
print(f"Genes shared in both  : {len(shared)}")
print("Only shared genes are used for the differential analysis so we")
print("are comparing the same set of genes across both conditions.")

rest_idx = [rest_genes.index(g) for g in shared]
act_idx  = [act_genes.index(g)  for g in shared]
G_rest_shared = G_rest[np.ix_(rest_idx, rest_idx)]
G_act_shared  = G_act[np.ix_(act_idx,  act_idx)]
shared_symbols = grn_rest.var["symbol_clean"].iloc[rest_idx].tolist()


# =============================================================================
# 2. NETWORK STATISTICS
# =============================================================================

print("\n" + "="*70)
print("SECTION 2: Network-level statistics")
print("="*70)
print("These metrics describe the overall structure of each regulatory network.")
print("Out-degree = number of other genes a gene regulates (how active a regulator it is).")
print("In-degree  = number of regulators pointing at a gene (how heavily regulated it is).")
print("Density    = fraction of all possible regulatory edges that are actually present.\n")

def network_stats(G, name):
    n            = G.shape[0]
    total_weight = float(G.sum())
    nonzero      = G[G > 0]
    mean_weight  = float(nonzero.mean()) if len(nonzero) > 0 else 0
    out_degree   = (G > 0).sum(axis=1)
    in_degree    = (G > 0).sum(axis=0)
    density      = float((G > 0).sum()) / (n * (n - 1))

    print(f"  [{name}]")
    print(f"    Total regulatory weight  : {total_weight:.2f}")
    print(f"      -> Higher total weight means more overall regulatory activity")
    print(f"    Mean nonzero edge weight : {mean_weight:.5f}")
    print(f"      -> Average strength of edges that exist")
    print(f"    Network density          : {density*100:.2f}% of all possible edges are active")
    print(f"    Max out-degree           : {out_degree.max()} — one gene regulates up to this many others")
    print(f"    Max in-degree            : {in_degree.max()} — one gene receives input from up to this many regulators")
    return out_degree, in_degree

out_rest, in_rest = network_stats(G_rest, "rest")
print()
out_act,  in_act  = network_stats(G_act,  "act")


# =============================================================================
# 3. TF ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("SECTION 3: Transcription factor regulatory activity")
print("="*70)
print("We sum the outgoing edge weights for each TF to get a regulatory score.")
print("A higher score means the TF drives more genes more strongly.")
print("Comparing scores between rest and act shows which TFs become")
print("more or less active during T cell activation.\n")

def tf_scores(G, grn, name):
    is_tf  = grn.var["isTF"].values
    syms   = grn.var["symbol_clean"].values
    scores = G.sum(axis=1)

    tf_df = pd.DataFrame({
        "symbol": syms[is_tf],
        "regulatory_score": scores[is_tf],
    }).sort_values("regulatory_score", ascending=False).reset_index(drop=True)

    print(f"  [{name}] Top {min(TOP_N, len(tf_df))} most active TFs by total outgoing weight:")
    if len(tf_df) == 0:
        print("    No TFs identified — check that gene_symbol column maps to known TF names.")
    else:
        print(tf_df.head(TOP_N).to_string(index=False))
    return tf_df

tf_rest = tf_scores(G_rest, grn_rest, "rest")
print()
tf_act  = tf_scores(G_act,  grn_act,  "act")


# =============================================================================
# 4. DIFFERENTIAL EDGES
# =============================================================================

print("\n" + "="*70)
print("SECTION 4: Differential regulatory edges (rest vs act)")
print("="*70)
print("We compute (act weight - rest weight) for every gene pair.")
print("Positive delta = a regulatory link that is NEW or STRONGER in activated cells.")
print("Negative delta = a regulatory link that DISAPPEARS or WEAKENS upon activation.")
print("These are the specific gene-gene regulatory relationships that change")
print("when a T cell transitions from resting to activated state.")
print(f"We only report edges where |delta| > {DIFF_THRESH} to focus on meaningful changes.\n")

G_diff = G_act_shared - G_rest_shared

n_shared = len(shared)
rows, cols = np.meshgrid(range(n_shared), range(n_shared), indexing="ij")
rows = rows.ravel(); cols = cols.ravel()
delta = G_diff.ravel()

mask = np.abs(delta) > DIFF_THRESH
diff_df = pd.DataFrame({
    "regulator": [shared_symbols[r] for r in rows[mask]],
    "target":    [shared_symbols[c] for c in cols[mask]],
    "delta":     delta[mask],
    "direction": ["gained" if d > 0 else "lost" for d in delta[mask]],
}).sort_values("delta", key=abs, ascending=False).reset_index(drop=True)

print(f"  Total rewired edges : {len(diff_df)}")
print(f"  Gained in activation: {(diff_df.direction == 'gained').sum()}")
print(f"    -> These are regulatory links that form when T cells are stimulated")
print(f"  Lost in activation  : {(diff_df.direction == 'lost').sum()}")
print(f"    -> These are regulatory links that dissolve upon stimulation")

print(f"\n  Top {TOP_N} GAINED edges — new regulatory links in activated T cells:")
print(diff_df[diff_df.direction == "gained"].head(TOP_N).to_string(index=False))

print(f"\n  Top {TOP_N} LOST edges — regulatory links that dissolve upon activation:")
print(diff_df[diff_df.direction == "lost"].head(TOP_N).to_string(index=False))


# =============================================================================
# 5. TOP REWIRED GENES
# =============================================================================

print("\n" + "="*70)
print("SECTION 5: Most rewired genes")
print("="*70)
print("We measure how much each gene's total incoming and outgoing regulatory")
print("weight changes between rest and act.")
print("out_delta > 0 means the gene REGULATES MORE genes in the activated state.")
print("in_delta  > 0 means the gene IS MORE HEAVILY REGULATED in the activated state.")
print("Genes with large changes in either direction are key players in the")
print("transcriptional reprogramming that occurs during T cell activation.\n")

out_delta = G_act_shared.sum(axis=1) - G_rest_shared.sum(axis=1)
in_delta  = G_act_shared.sum(axis=0) - G_rest_shared.sum(axis=0)

rewired_df = pd.DataFrame({
    "symbol":          shared_symbols,
    "out_delta":       out_delta,
    "in_delta":        in_delta,
    "abs_total_delta": np.abs(out_delta) + np.abs(in_delta),
}).sort_values("abs_total_delta", ascending=False).reset_index(drop=True)

print(f"  Top {TOP_N} most rewired genes overall:")
print(rewired_df.head(TOP_N).to_string(index=False))

print(f"\n  Genes that become STRONGER REGULATORS upon activation (top {TOP_N}):")
print(f"  -> These likely include key TFs driving the activation program")
print(rewired_df.nlargest(TOP_N, "out_delta")[["symbol","out_delta"]].to_string(index=False))

print(f"\n  Genes that become MORE HEAVILY REGULATED upon activation (top {TOP_N}):")
print(f"  -> These are downstream targets that get switched on during activation")
print(rewired_df.nlargest(TOP_N, "in_delta")[["symbol","in_delta"]].to_string(index=False))


# =============================================================================
# 6. HUB GENE COMPARISON
# =============================================================================

print("\n" + "="*70)
print("SECTION 6: Hub gene comparison")
print("="*70)
print("Hub genes have the highest total outgoing regulatory weight —")
print("they are the master regulators of each cell state.")
print("A gene that is a hub in act but not rest is likely driving activation.")
print("A gene that is a hub in rest but not act is likely a homeostasis maintainer.\n")

def hub_genes(G, symbols, name, n=TOP_N):
    scores  = G.sum(axis=1)
    top_idx = np.argsort(scores)[::-1][:n]
    df = pd.DataFrame({
        "symbol": [symbols[i] for i in top_idx],
        "total_outgoing_weight": scores[top_idx],
    })
    print(f"  [{name}] Top {n} hub regulators:")
    print(df.to_string(index=False))
    return set(df["symbol"].tolist())

hubs_rest = hub_genes(G_rest_shared, shared_symbols, "rest")
print()
hubs_act  = hub_genes(G_act_shared,  shared_symbols, "act")

activation_specific = hubs_act - hubs_rest
rest_specific       = hubs_rest - hubs_act
shared_hubs         = hubs_rest & hubs_act

print(f"\n  Hubs in BOTH states (stable core regulators): {sorted(shared_hubs)}")
print(f"    -> These genes maintain regulatory control regardless of activation state")
print(f"  Hubs GAINED in activation (activation drivers): {sorted(activation_specific)}")
print(f"    -> These genes become dominant regulators specifically when T cells are stimulated")
print(f"  Hubs LOST in activation (rest-specific regulators): {sorted(rest_specific)}")
print(f"    -> These genes regulate the resting state and are suppressed upon activation")


# =============================================================================
# 7. VISUALISATION
# =============================================================================

print("\n" + "="*70)
print("SECTION 7: Generating plots")
print("="*70)

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(G_rest[G_rest > 0].ravel(), bins=80, alpha=0.6,
         color="steelblue", density=True, label="rest")
ax1.hist(G_act[G_act > 0].ravel(),  bins=80, alpha=0.6,
         color="darkorange", density=True, label="act")
ax1.set_title("Edge weight distribution\n(nonzero edges only)", fontsize=11)
ax1.set_xlabel("Attention weight"); ax1.set_ylabel("Density")
ax1.legend()
ax1.text(0.02, 0.95,
         "Compares whether one state has\ngenerally stronger regulatory links.",
         transform=ax1.transAxes, fontsize=8, va="top", color="gray")

ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(G_diff.ravel(), bins=100, color="mediumpurple", alpha=0.8)
ax2.axvline(0, color="black", linewidth=1)
ax2.axvline( DIFF_THRESH, color="red", linewidth=1, linestyle="--",
             label=f"threshold ±{DIFF_THRESH}")
ax2.axvline(-DIFF_THRESH, color="red", linewidth=1, linestyle="--")
ax2.set_title("Edge weight changes\n(act - rest)", fontsize=11)
ax2.set_xlabel("Delta weight"); ax2.set_ylabel("Count")
ax2.legend(fontsize=8)
ax2.text(0.02, 0.95,
         "Right of 0 = edges gained in activation.\nLeft = edges lost.",
         transform=ax2.transAxes, fontsize=8, va="top", color="gray")

ax3 = fig.add_subplot(gs[0, 2])
top_rw = rewired_df.head(15)
colors = ["darkorange" if d > 0 else "steelblue" for d in top_rw["out_delta"]]
ax3.barh(top_rw["symbol"][::-1], top_rw["out_delta"][::-1], color=colors[::-1])
ax3.axvline(0, color="black", linewidth=0.8)
ax3.set_title("Top 15 rewired regulators\n(outgoing weight delta)", fontsize=11)
ax3.set_xlabel("Change in outgoing weight (act - rest)")
ax3.text(0.02, 0.02,
         "Orange = more active in act.\nBlue = more active in rest.",
         transform=ax3.transAxes, fontsize=8, va="bottom", color="gray")

ax4 = fig.add_subplot(gs[1, 0])
all_hubs   = sorted(hubs_rest | hubs_act)
rest_sc    = {s: G_rest_shared.sum(axis=1)[shared_symbols.index(s)]
              for s in all_hubs if s in shared_symbols}
act_sc     = {s: G_act_shared.sum(axis=1)[shared_symbols.index(s)]
              for s in all_hubs if s in shared_symbols}
hub_df     = pd.DataFrame({"rest": rest_sc, "act": act_sc}).fillna(0)
x = np.arange(len(hub_df)); w = 0.35
ax4.bar(x - w/2, hub_df["rest"], w, label="rest",  color="steelblue",  alpha=0.8)
ax4.bar(x + w/2, hub_df["act"],  w, label="act",   color="darkorange", alpha=0.8)
ax4.set_xticks(x)
ax4.set_xticklabels(hub_df.index, rotation=45, ha="right", fontsize=8)
ax4.set_title("Hub gene regulatory scores\n(rest vs act)", fontsize=11)
ax4.set_ylabel("Total outgoing weight")
ax4.legend()
ax4.text(0.02, 0.95,
         "Taller bar = stronger regulatory\ninfluence in that state.",
         transform=ax4.transAxes, fontsize=8, va="top", color="gray")

ax5 = fig.add_subplot(gs[1, 1])
top_gained = diff_df[diff_df.direction == "gained"].head(15)
if len(top_gained) > 0:
    pivot_g = top_gained.pivot_table(
        index="regulator", columns="target", values="delta", fill_value=0)
    sns.heatmap(pivot_g, ax=ax5, cmap="Oranges", cbar_kws={"shrink": 0.6})
    ax5.set_title("Top gained edges (act - rest)\nRegulator -> Target", fontsize=11)
    ax5.tick_params(axis="both", labelsize=7)
else:
    ax5.text(0.5, 0.5, "No gained edges\nabove threshold",
             ha="center", va="center", transform=ax5.transAxes)
    ax5.set_title("Top gained edges", fontsize=11)
ax5.text(0.02, -0.28,
         "New/strengthened regulatory links\nin activated T cells.",
         transform=ax5.transAxes, fontsize=8, color="gray")

ax6 = fig.add_subplot(gs[1, 2])
top_lost = diff_df[diff_df.direction == "lost"].head(15)
if len(top_lost) > 0:
    pivot_l = top_lost.pivot_table(
        index="regulator", columns="target", values="delta", fill_value=0)
    sns.heatmap(pivot_l.abs(), ax=ax6, cmap="Blues", cbar_kws={"shrink": 0.6})
    ax6.set_title("Top lost edges (rest - act)\nRegulator -> Target", fontsize=11)
    ax6.tick_params(axis="both", labelsize=7)
else:
    ax6.text(0.5, 0.5, "No lost edges\nabove threshold",
             ha="center", va="center", transform=ax6.transAxes)
    ax6.set_title("Top lost edges", fontsize=11)
ax6.text(0.02, -0.28,
         "Regulatory links that dissolve\nwhen T cells are activated.",
         transform=ax6.transAxes, fontsize=8, color="gray")

fig.suptitle("scPRINT2 GRN Analysis: Resting vs Activated T Cells",
             fontsize=15, fontweight="bold", y=1.01)

plot_path = os.path.join(SAVE_DIR, "grn_analysis.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"  Plot saved to: {plot_path}")


# =============================================================================
# 8. SAVE SUMMARY TABLES
# =============================================================================

print("\n" + "="*70)
print("SECTION 8: Saving summary tables")
print("="*70)

diff_path    = os.path.join(SAVE_DIR, "differential_edges.csv")
rewired_path = os.path.join(SAVE_DIR, "rewired_genes.csv")
tf_path      = os.path.join(SAVE_DIR, "tf_scores.csv")

diff_df.to_csv(diff_path, index=False)
rewired_df.to_csv(rewired_path, index=False)

tf_combined = (
    tf_rest.rename(columns={"regulatory_score": "score_rest"})
    .merge(tf_act.rename(columns={"regulatory_score": "score_act"}),
           on="symbol", how="outer")
    .fillna(0)
)
tf_combined["delta"] = tf_combined["score_act"] - tf_combined["score_rest"]
tf_combined = tf_combined.sort_values("delta", key=abs, ascending=False)
tf_combined.to_csv(tf_path, index=False)

print(f"  differential_edges.csv -> all rewired edges with direction and delta weight")
print(f"  rewired_genes.csv      -> per-gene summary of how much regulation changed")
print(f"  tf_scores.csv          -> TF regulatory scores in rest and act with delta")
print(f"\n  Files saved to: {SAVE_DIR}/")

print("\n" + "="*70)
print("DONE — Key findings summary")
print("="*70)
print(f"  Shared genes analysed         : {len(shared)}")
print(f"  Total rewired edges           : {len(diff_df)}")
print(f"  Edges gained in activation    : {(diff_df.direction == 'gained').sum()}")
print(f"  Edges lost in activation      : {(diff_df.direction == 'lost').sum()}")
print(f"  Activation-specific hub genes : {sorted(activation_specific)}")
print(f"  Stable hub genes (both states): {sorted(shared_hubs)}")
print(f"  Hub genes lost in activation  : {sorted(rest_specific)}")
print("\nRecommended next step: cross-reference activation-specific hub genes")
print("against known T cell activation TFs (NFAT, AP-1, NF-kB) to validate")
print("whether scPRINT recovers known biology from attention weights alone.")